# ML Honors Assignment - 4

**Name:** Vaishnav Mankar

**Rollno:** 41252

---

**Problem Statement:** Text classification for Sentimental analysis using KNN. Note: Use twitter data.

**Dataset:** Twitter US Airline Sentiment (Kaggle — `Tweets.csv`)

**Requirements:** Python 3, KaggleHub, Pandas, Scikit-Learn

**Code:-**

In [ ]:
!pip install -q kagglehub

In [ ]:
import re
import pandas as pd
import kagglehub
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

path = kagglehub.dataset_download("crowdflower/twitter-airline-sentiment")
df = pd.read_csv(f"{path}/Tweets.csv")[["text", "airline_sentiment"]]
print("Dataset Shape:", df.shape)
print(df["airline_sentiment"].value_counts())
display(df.head())

In [ ]:
def clean(text):
    text = re.sub(r"http\S+|@\w+", " ", text.lower())
    return re.sub(r"[^a-z\s]", " ", text)


df["clean"] = df["text"].apply(clean)
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["airline_sentiment"], test_size=0.2, stratify=df["airline_sentiment"], random_state=42)

tfidf = TfidfVectorizer(stop_words="english", min_df=2, max_features=5000)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)
print("TF-IDF matrix:", X_train_vec.shape)

In [ ]:
scores = {}
for k in (1, 3, 5, 7, 11, 15):
    knn = KNeighborsClassifier(n_neighbors=k, metric="cosine")
    scores[k] = cross_val_score(knn, X_train_vec, y_train, cv=3).mean()
    print(f"K = {k:>2}: cross-validation accuracy {scores[k]:.3f}")
best_k = max(scores, key=scores.get)
print("Best K:", best_k)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k, metric="cosine").fit(X_train_vec, y_train)
pred = knn.predict(X_test_vec)
print(f"Test Accuracy: {accuracy_score(y_test, pred) * 100:.2f}%\n")
print(classification_report(y_test, pred))

In [ ]:
samples = [
    "Great flight, the crew was lovely and friendly",
    "Worst airline ever, my bag was lost and the delay was awful",
    "What time does the flight to Delhi board",
]
for tweet, label in zip(samples, knn.predict(tfidf.transform([clean(t) for t in samples]))):
    print(f"{label:>9}  <-  {tweet}")